1# Install packages

In [ ]:
!pip install requests
!pip install beautifulsoup4

2#Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import sqlite3
import re
from pathlib import Path
print("Libraries imported successfully.")

Libraries imported successfully.


3#Configuration

In [ ]:
BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"

GBP_TO_INR = 105.50

OUTPUT_DIR = Path(".")
DB_PATH = OUTPUT_DIR / "books.db"
CSV_PATH = OUTPUT_DIR / "books_cleaned.csv"

print(f"Fixed conversion rate: 1 GBP = {GBP_TO_INR} INR")

Fixed conversion rate: 1 GBP = 105.5 INR


4# Scrape the first 5 catalogue pages

In [ ]:
def fetch_page(url):
    """
    Download a web page and return its BeautifulSoup object.
    Raises an error for unsuccessful HTTP responses.
    """

    response = requests.get(
        url,
        timeout=15,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")

5 — Test the connection

In [ ]:
soup = fetch_page(START_URL)

print("Page title:", soup.title.get_text(strip=True))

Page title: All products | Books to Scrape - Sandbox


6. Extract books

In [ ]:
def scrape_listing_page(url):
    soup = fetch_page(url)

    books = []

    for article in soup.select("article.product_pod"):

        title_element = article.select_one("h3 a")
        price_element = article.select_one(".price_color")
        rating_element = article.select_one(".star-rating")
        availability_element = article.select_one(".availability")

        title = title_element.get("title", "").strip()
        price = price_element.get_text(strip=True)
        rating = " ".join(rating_element.get("class", []))
        availability = availability_element.get_text(" ", strip=True)

        books.append({
            "title": title,
            "price": price,
            "star_rating": rating,
            "availability": availability
        })

    return books

7. Scrape 5 pages

In [ ]:
all_books = []

for page_number in range(1, 6):

    url = f"https://books.toscrape.com/catalogue/page-{page_number}.html"

    page_books = scrape_listing_page(url)

    all_books.extend(page_books)

    print(
        f"Page {page_number}: "
        f"{len(page_books)} books scraped"
    )

print()
print("Total books scraped:", len(all_books))

Page 1: 20 books scraped
Page 2: 20 books scraped
Page 3: 20 books scraped
Page 4: 20 books scraped
Page 5: 20 books scraped

Total books scraped: 100


8. Add category

In [ ]:
for i, book in enumerate(all_books):

    book["category"] = get_category(book["product_url"])

    if (i + 1) % 10 == 0:
        print(f"Processed category for {i + 1}/{len(all_books)} books")

Processed category for 10/100 books
Processed category for 20/100 books
Processed category for 30/100 books
Processed category for 40/100 books
Processed category for 50/100 books
Processed category for 60/100 books
Processed category for 70/100 books
Processed category for 80/100 books
Processed category for 90/100 books
Processed category for 100/100 books


In [ ]:
def scrape_listing_page(url):
    soup = fetch_page(url)

    books = []

    for article in soup.select("article.product_pod"):

        title_element = article.select_one("h3 a")
        price_element = article.select_one(".price_color")
        rating_element = article.select_one(".star-rating")
        availability_element = article.select_one(".availability")

        title = title_element.get("title", "").strip()
        price = price_element.get_text(strip=True)
        rating = " ".join(rating_element.get("class", []))
        availability = availability_element.get_text(" ", strip=True)

        relative_url = title_element.get("href")

        product_url = (
            "https://books.toscrape.com/catalogue/"
            + relative_url.replace("../", "")
        )

        books.append({
            "title": title,
            "price": price,
            "star_rating": rating,
            "availability": availability,
            "product_url": product_url
        })

    return books

9. Extract category from product pages

In [ ]:
def get_category(product_url):

    soup = fetch_page(product_url)

    breadcrumb = soup.select("ul.breadcrumb li a")

    if len(breadcrumb) >= 3:
        return breadcrumb[2].get_text(strip=True)

    return None

Add categories

10. Create DataFrame

In [ ]:
df_raw = pd.DataFrame(all_books)

df_raw.head()

,title,price,star_rating,availability,product_url,category
0,A Light in the Attic,Â£51.77,star-rating Three,In stock,https://books.toscrape.com/catalogue/a-light-i...,Poetry
1,Tipping the Velvet,Â£53.74,star-rating One,In stock,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction
2,Soumission,Â£50.10,star-rating One,In stock,https://books.toscrape.com/catalogue/soumissio...,Fiction
3,Sharp Objects,Â£47.82,star-rating Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,star-rating Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...,History


Check the dataset:

In [ ]:
print("Rows:", len(df_raw))
print("Columns:", df_raw.columns.tolist())

Rows: 100
Columns: ['title', 'price', 'star_rating', 'availability', 'product_url', 'category']


11. Check categories

In [ ]:
print(
    df_raw["category"]
    .value_counts()
)

category
Sequential Art        14
Nonfiction            12
Default                9
Poetry                 7
Fiction                5
Food and Drink         5
Add a comment          5
Young Adult            4
History                4
Fantasy                4
Mystery                3
Music                  3
Childrens              3
Thriller               3
Philosophy             2
Romance                2
Spirituality           2
Science Fiction        2
Politics               1
Business               1
Historical Fiction     1
Travel                 1
Art                    1
Contemporary           1
New Adult              1
Science                1
Health                 1
Horror                 1
Self Help              1
Name: count, dtype: int64


In [ ]:
print(
    "Number of categories:",
    df_raw["category"].nunique()
)

Number of categories: 29


Clean price-The website displays prices such as:

£51.77

We need:

51.77

In [ ]:
def parse_price(value):

    try:
        cleaned = re.sub(r"[^\d.]", "", str(value))
        return float(cleaned)

    except (ValueError, TypeError):
        return None

In [ ]:
df_raw["price_gbp"] = df_raw["price"].apply(parse_price)

Clean star rating

The website gives values such as:

star-rating Three

In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}


def parse_rating(value):

    try:
        rating_word = str(value).split()[-1]
        return rating_map.get(rating_word)

    except (AttributeError, IndexError):
        return None


df_raw["rating"] = df_raw["star_rating"].apply(parse_rating)

In [ ]:
print(df_raw["rating"].value_counts().sort_index())

rating
1    22
2    19
3    22
4    18
5    19
Name: count, dtype: int64


14. Parse availability

The website gives values such as:

In stock (22 available)

We need:

True

In [ ]:
def parse_stock(value):

    text = str(value).strip().lower()

    if text.startswith("in stock"):
        return True

    if text.startswith("out of stock"):
        return False

    return None


df_raw["in_stock"] = df_raw["availability"].apply(parse_stock)

15. Handle failed parsing

The assignment specifically says the pipeline must not crash on messy rows.

For numeric fields, use median imputation.

In [ ]:
# Numeric fields: median imputation

price_median = df_raw["price_gbp"].median()
rating_median = df_raw["rating"].median()

df_raw["price_gbp"] = (
    df_raw["price_gbp"]
    .fillna(price_median)
)

df_raw["rating"] = (
    df_raw["rating"]
    .fillna(round(rating_median))
    .astype(int)
)

In [ ]:
# Non-numeric critical fields:
# drop rows where title, category, or stock status cannot be parsed.

before_cleaning = len(df_raw)

df_clean = df_raw.dropna(
    subset=[
        "title",
        "category",
        "in_stock"
    ]
).copy()

after_cleaning = len(df_clean)

print("Rows before cleaning:", before_cleaning)
print("Rows after cleaning:", after_cleaning)
print("Rows removed:", before_cleaning - after_cleaning)

Rows before cleaning: 100
Rows after cleaning: 100
Rows removed: 0


16. Convert GBP → INR

In [ ]:
df_clean["price_inr"] = (
    df_clean["price_gbp"] * GBP_TO_INR
).round(2)

In [ ]:
df_clean[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].head()

,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.74,3,True,Poetry
1,Tipping the Velvet,53.74,5669.57,1,True,Historical Fiction
2,Soumission,50.10,5285.55,1,True,Fiction
3,Sharp Objects,47.82,5045.01,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History


17. Select final columns

In [ ]:
df_final = df_clean[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()

18. Verify data types

In [ ]:
print(df_final.dtypes)

title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [ ]:
assert df_final["price_gbp"].dtype == "float64"
assert df_final["price_inr"].dtype == "float64"
assert df_final["rating"].between(1, 5).all()
assert df_final["in_stock"].dtype == bool

print("Data validation passed.")

Data validation passed.


19. Verify at least 60 books

In [ ]:
assert len(df_final) >= 60, "Dataset contains fewer than 60 books."

print("Total cleaned books:", len(df_final))
print("Total categories:", df_final["category"].nunique())

Total cleaned books: 100
Total categories: 29


20. Save cleaned CSV

In [ ]:
df_final.to_csv(
    CSV_PATH,
    index=False
)

print(f"Saved: {CSV_PATH}")

Saved: books_cleaned.csv


21. Create SQLite database

This is a key grading requirement.

We need two normalized tables:

categories
     │
     │ category_id
     ↓
books

In [ ]:
conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);
""")

conn.commit()

print("SQLite schema created.")

SQLite schema created.


22. Insert categories

In [ ]:
categories_df = (
    df_final[["category"]]
    .drop_duplicates()
    .sort_values("category")
    .reset_index(drop=True)
)

categories_df["category_id"] = (
    categories_df.index + 1
)

categories_df = categories_df[
    ["category_id", "category"]
]

categories_df

,category_id,category
0,1,Add a comment
1,2,Art
2,3,Business
3,4,Childrens
4,5,Contemporary
5,6,Default
6,7,Fantasy
7,8,Fiction
8,9,Food and Drink
9,10,Health


In [ ]:
categories_df.rename(
    columns={"category": "category_name"}
).to_sql(
    "categories",
    conn,
    if_exists="append",
    index=False
)

29

23. Create category lookup

In [ ]:
  category_lookup = dict(
      zip(
          categories_df["category"],
          categories_df["category_id"]
      )
  )

  df_final["category_id"] = (
      df_final["category"]
      .map(category_lookup)
  )

24. Insert books

In [ ]:
books_to_insert = df_final[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

books_to_insert["in_stock"] = (
    books_to_insert["in_stock"]
    .astype(int)
)

books_to_insert.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

conn.commit()

print("Books inserted into SQLite.")

Books inserted into SQLite.


25. Verify database

In [ ]:
print(
    pd.read_sql(
        "SELECT COUNT(*) AS book_count FROM books;",
        conn
    )
)

print(
    pd.read_sql(
        "SELECT COUNT(*) AS category_count FROM categories;",
        conn
    )
)

   book_count
0         100
   category_count
0              29


26. SQL Query 1 — SELECT + WHERE
This satisfies the SELECT and WHERE requirement.

In [ ]:
query_1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4;
"""

result_1 = pd.read_sql(query_1, conn)

print(query_1)
display(result_1.head(10))


SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4;



,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4
5,Set Me Free,17.46,5
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
7,Rip it Up and Start Again,35.02,5
8,Chase Me (Paris Nights #2),25.27,5
9,Black Dust,34.53,5


27. SQL Query 2 — ORDER BY + LIMIT

In [ ]:
query_2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

result_2 = pd.read_sql(query_2, conn)

print(query_2)
display(result_2)


SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;



,title,price_gbp,rating
0,The Death of Humanity: and the Case for Life,58.11,4
1,Slow States of Collapse: Poems,57.31,3
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3
3,The Past Never Ends,56.50,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
5,Masks and Shadows,56.40,2
6,The Secret of Dreadwillow Carse,56.13,1
7,The Electric Pencil: Drawings from Inside Stat...,56.06,1
8,Birdsong: A Story in Pictures,54.64,3
9,Sapiens: A Brief History of Humankind,54.23,5


28. SQL Query 3 — DISTINCT

In [ ]:
query_3 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating;
"""

result_3 = pd.read_sql(query_3, conn)

print(query_3)
display(result_3)


SELECT DISTINCT rating
FROM books
ORDER BY rating;



,rating
0,1
1,2
2,3
3,4
4,5


29. SQL Query 4 — BETWEEN

In [ ]:
query_4 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp;
"""

result_4 = pd.read_sql(query_4, conn)

print(query_4)
display(result_4)


SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp;



,title,price_gbp,price_inr
0,The Inefficiency Assassin: Time Management Tac...,20.59,2172.24
1,Shakespeare's Sonnets,20.66,2179.63
2,In the Country We Love: My Family Divided,22.00,2321.00
3,America's Cradle of Quarterbacks: Western Penn...,22.50,2373.75
4,The Boys in the Boat: Nine Americans and Their...,22.60,2384.30
5,The Requiem Red,22.65,2389.57
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10
7,The Elephant Tree,23.82,2513.01
8,Olio,23.88,2519.34
9,The Mindfulness and Acceptance Workbook for An...,23.89,2520.40


30. SQL Query 5 — IN

In [ ]:
query_5 = """
SELECT title, rating, category_id
FROM books
WHERE rating IN (4, 5)
ORDER BY rating DESC;
"""

result_5 = pd.read_sql(query_5, conn)

print(query_5)
display(result_5.head(10))


SELECT title, rating, category_id
FROM books
WHERE rating IN (4, 5)
ORDER BY rating DESC;



,title,rating,category_id
0,Sapiens: A Brief History of Humankind,5,12
1,Set Me Free,5,29
2,Scott Pilgrim's Precious Little Life (Scott Pi...,5,25
3,Rip it Up and Start Again,5,14
4,Chase Me (Paris Nights #2),5,21
5,Black Dust,5,21
6,Worlds Elsewhere: Journeys Around Shakespeareâ...,5,17
7,The Four Agreements: A Practical Guide to Pers...,5,26
8,The Elephant Tree,5,27
9,Sophie's World,5,18


31. SQL Query 6 — JOIN

This is the important normalized database demonstration.

This satisfies the JOIN requirement as well as ORDER BY and LIMIT.

In [ ]:
query_6 = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""

result_6_sql = pd.read_sql(query_6, conn)

print(query_6)
display(result_6_sql)


SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;



,title,category_name,price_gbp,price_inr,rating,in_stock
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5,1
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5,1
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5,1
3,Private Paris (Private #10),Fiction,47.61,5022.85,5,1
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5,1
5,Join,Science Fiction,35.67,3763.19,5,1
6,Rip it Up and Start Again,Music,35.02,3694.61,5,1
7,Black Dust,Romance,34.53,3642.92,5,1
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5,1
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5,1


32. Read at least two SQL results with pd.read_sql

In [ ]:
result_1 = pd.read_sql(query_1, conn)
result_2 = pd.read_sql(query_2, conn)

In [ ]:
sql_result_a = pd.read_sql(query_1, conn)
sql_result_b = pd.read_sql(query_2, conn)

print("Query 1 result:")
display(sql_result_a.head())

print("Query 2 result:")
display(sql_result_b.head())

Query 1 result:


,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4


Query 2 result:


,title,price_gbp,rating
0,The Death of Humanity: and the Case for Life,58.11,4
1,Slow States of Collapse: Poems,57.31,3
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3
3,The Past Never Ends,56.50,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1


33. Reproduce JOIN using pd.merge

This is another important grading requirement.

We need to reproduce the SQL JOIN without SQL.

In [ ]:
books_memory = df_final[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

categories_memory = categories_df.rename(
    columns={
        "category": "category_name"
    }
)

merged_result = pd.merge(
    books_memory,
    categories_memory,
    on="category_id",
    how="inner"
)

merged_result = merged_result[
    [
        "title",
        "category_name",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock"
    ]
]

merged_result = (
    merged_result
    .sort_values(
        ["rating", "price_gbp"],
        ascending=[False, False]
    )
    .head(10)
)

display(merged_result)

,title,category_name,price_gbp,price_inr,rating,in_stock
4,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5,True
13,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5,True
46,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5,True
42,Private Paris (Private #10),Fiction,47.61,5022.85,5,True
28,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5,True
98,Join,Science Fiction,35.67,3763.19,5,True
14,Rip it Up and Start Again,Music,35.02,3694.61,5,True
24,Black Dust,Romance,34.53,3642.92,5,True
72,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5,True
23,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5,True


34. Compare SQL JOIN vs pandas JOIN

In [ ]:
sql_join_comparison = result_6_sql.copy()

pandas_join_comparison = merged_result.copy()

sql_join_comparison = sql_join_comparison.reset_index(drop=True)
pandas_join_comparison = pandas_join_comparison.reset_index(drop=True)

print("SQL JOIN result:")
display(sql_join_comparison)

print("Pandas merge result:")
display(pandas_join_comparison)

SQL JOIN result:


,title,category_name,price_gbp,price_inr,rating,in_stock
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5,1
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5,1
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5,1
3,Private Paris (Private #10),Fiction,47.61,5022.85,5,1
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5,1
5,Join,Science Fiction,35.67,3763.19,5,1
6,Rip it Up and Start Again,Music,35.02,3694.61,5,1
7,Black Dust,Romance,34.53,3642.92,5,1
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5,1
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5,1


Pandas merge result:


,title,category_name,price_gbp,price_inr,rating,in_stock
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5,True
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5,True
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5,True
3,Private Paris (Private #10),Fiction,47.61,5022.85,5,True
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5,True
5,Join,Science Fiction,35.67,3763.19,5,True
6,Rip it Up and Start Again,Music,35.02,3694.61,5,True
7,Black Dust,Romance,34.53,3642.92,5,True
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5,True
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5,True


In [ ]:
comparison = sql_join_comparison.equals(
    pandas_join_comparison
)

print("Do SQL JOIN and pandas merge produce equivalent results?", comparison)

Do SQL JOIN and pandas merge produce equivalent results? False


35. Database schema verification

In [ ]:
schema_query = """
SELECT
    name,
    sql
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

schema = pd.read_sql(schema_query, conn)

display(schema)

,name,sql
0,books,CREATE TABLE books (\n book_id INTEGER PRIM...
1,categories,CREATE TABLE categories (\n category_id INT...
2,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"


36. Foreign-key verification

This gives you evidence that books.category_id references categories.category_id.

In [ ]:
foreign_key_query = """
PRAGMA foreign_key_list(books);
"""

foreign_keys = pd.read_sql(
    foreign_key_query,
    conn
)

display(foreign_keys)

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,categories,category_id,category_id,NO ACTION,NO ACTION,NONE


37. Final validation

In [ ]:
print("========== FINAL VALIDATION ==========")

print("Total books:", len(df_final))
print("Total categories:", df_final["category"].nunique())

print(
    "Price GBP type:",
    df_final["price_gbp"].dtype
)

print(
    "Price INR type:",
    df_final["price_inr"].dtype
)

print(
    "Rating type:",
    df_final["rating"].dtype
)

print(
    "In-stock type:",
    df_final["in_stock"].dtype
)

print(
    "GBP → INR rate:",
    GBP_TO_INR
)

print(
    "SQL/Pandas JOIN equivalent:",
    comparison
)

========== FINAL VALIDATION ==========
Total books: 100
Total categories: 29
Price GBP type: float64
Price INR type: float64
Rating type: int64
In-stock type: bool
GBP → INR rate: 105.5
SQL/Pandas JOIN equivalent: False


38. Close the database

In [ ]:
conn.close()

print("Database connection closed.")

Database connection closed.
